In [4]:
import numpy as np
import pandas as pd
import os
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
import numpy as np

load_dotenv(find_dotenv())

engine = create_engine(URL.create(
    "postgresql+psycopg2",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host="localhost",
    port=int(os.getenv("DB_PORT", "5432")),
    database=os.getenv("DB_NAME"),
))

df = pd.read_parquet('../data/application_clean.parquet')
print(df.shape)

(307507, 86)


In [7]:
### splitting the data into three parts 60/20/20
#60% for learning
#20% for building EV metric
#20# for testing

from sklearn.model_selection import train_test_split

ids = df['sk_id_curr']
X = df.drop(columns=['target', 'sk_id_curr'])
y = df['target']

# 1) peel off the test set — 20%
X_temp, X_test, y_temp, y_test, id_temp, id_test = train_test_split(
    X, y, ids, test_size=0.20, stratify=y, random_state=42)

# 2) split what's left into train and validation — 25% of 80% = 20% overall
X_train, X_val, y_train, y_val, id_train, id_val = train_test_split(
    X_temp, y_temp, id_temp, test_size=0.25, stratify=y_temp, random_state=42)

print("train", X_train.shape, round(y_train.mean(), 4))
print("val  ", X_val.shape,   round(y_val.mean(), 4))
print("test ", X_test.shape,  round(y_test.mean(), 4))

train (184503, 84) 0.0807
val   (61502, 84) 0.0807
test  (61502, 84) 0.0807


In [14]:
cat_cols = X.select_dtypes('object').columns.tolist()
num_cols = X.select_dtypes('number').columns.tolist()

print(len(cat_cols), "categorical")
print(len(num_cols), "numeric")
print("target in X:", 'target' in X.columns)
print("sk_id_curr in X:", 'sk_id_curr' in X.columns)
print(len(cat_cols) + len(num_cols), X.shape[1])

13 categorical
71 numeric
target in X: False
sk_id_curr in X: False
84 84


In [16]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

num_imputer = SimpleImputer(strategy="median")

num_scaler = StandardScaler()

# the two, in order
numeric_pipe = Pipeline([
    ("impute", num_imputer),
    ("scale",  num_scaler),
])

,missing_values,nan
,strategy,'median'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False


In [19]:
from sklearn.preprocessing import OneHotEncoder

cat_imputer = SimpleImputer(strategy="most_frequent")

cat_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

categorical_pipe = Pipeline([
    ("impute", cat_imputer),
    ("encode", cat_encoder),
])

In [21]:
from sklearn.compose import ColumnTransformer

preprocess = ColumnTransformer([
    ("num", numeric_pipe,     num_cols),
    ("cat", categorical_pipe, cat_cols),
])

In [23]:
X_train_prepared = preprocess.fit_transform(X_train)
print(X_train_prepared.shape)
print(np.isnan(X_train_prepared).sum())    # 0 — imputation worked
print(X_train_prepared[:, 0].mean())       # ~0 — scaling worked

(184503, 212)
0
2.9730627253041814e-17


In [24]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

logreg = Pipeline([
    ("prep",  preprocess),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced", n_jobs=-1)),
])

logreg.fit(X_train, y_train)

,steps,"[('prep', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [25]:
val_probs = logreg.predict_proba(X_val)[:, 1]
print("PR-AUC:", round(average_precision_score(y_val, val_probs), 4))
print("baseline:", round(y_val.mean(), 4))

PR-AUC: 0.2167
baseline: 0.0807


In [26]:
from sklearn.preprocessing import KBinsDiscretizer

bin_cols = ['amt_goods_price']

binner = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("bin",    KBinsDiscretizer(n_bins=10, encode="onehot-dense", strategy="quantile")),
])

preprocess_binned = ColumnTransformer([
    ("num", numeric_pipe,     [c for c in num_cols if c not in bin_cols]),
    ("cat", categorical_pipe, cat_cols),
    ("bin", binner,           bin_cols),
])

logreg_binned = Pipeline([
    ("prep",  preprocess_binned),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced", n_jobs=-1)),
])

logreg_binned.fit(X_train, y_train)
probs = logreg_binned.predict_proba(X_val)[:, 1]
print("binned PR-AUC:", round(average_precision_score(y_val, probs), 4))

C:\Users\Maria Pilipenko\anaconda3\Lib\site-packages\sklearn\preprocessing\_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(


binned PR-AUC: 0.2191


In [29]:
from xgboost import XGBClassifier

xgb_prep = ColumnTransformer([
    ("cat", cat_encoder, cat_cols),
], remainder="passthrough")

xgb = Pipeline([
    ("prep", xgb_prep),
    ("model", XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        scale_pos_weight=11.4,
        eval_metric="aucpr",
        tree_method="hist",
        n_jobs=-1,
        random_state=42,
    )),
])

In [30]:
%%time
xgb.fit(X_train, y_train)

CPU times: total: 1min 20s
Wall time: 6.86 s


,steps,"[('prep', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [31]:
val_probs_xgb = xgb.predict_proba(X_val)[:, 1]
print("XGB PR-AUC:", round(average_precision_score(y_val, val_probs_xgb), 4))
print("logreg     :", 0.2167)

XGB PR-AUC: 0.2335
logreg     : 0.2167


In [32]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "model__max_depth":        [3, 4, 5, 6, 8],
    "model__learning_rate":    [0.01, 0.03, 0.05, 0.1],
    "model__subsample":        [0.6, 0.8, 1.0],
    "model__colsample_bytree": [0.6, 0.8, 1.0],
    "model__min_child_weight": [1, 5, 10, 20],
    "model__reg_lambda":       [1, 5, 10],
}

search = RandomizedSearchCV(
    xgb,
    param_distributions=param_dist,
    n_iter=20,
    cv=3,
    scoring="average_precision",
    n_jobs=-1,
    random_state=42,
    verbose=2,
)

search.fit(X_train, y_train)

print(round(search.best_score_, 4))
print(search.best_params_)

Fitting 3 folds for each of 20 candidates, totalling 60 fits
0.2435
{'model__subsample': 0.8, 'model__reg_lambda': 1, 'model__min_child_weight': 10, 'model__max_depth': 4, 'model__learning_rate': 0.03, 'model__colsample_bytree': 0.8}
